# Notebook 03: Gold Dimensional Model - Star Schema

## Overview
This notebook creates a star schema dimensional model in the Gold layer for Power BI analytics.

## Prerequisites
- Notebook 02 completed successfully
- Silver tables: `report_summary`, `country_weekly`
- Reference data: `country_codes_iso3166.csv`, `au_regions.csv`, `epi_week_calendar.csv`

## Inputs
- Silver Delta tables from Notebook 02
- Reference dimension data

## Outputs
- Dimension: `gold.dim_country` (country master data)
- Dimension: `gold.dim_date` (date/epi week calendar)
- Dimension: `gold.dim_report` (report metadata)
- Fact: `gold.fact_cholera_cases` (case metrics)
- Fact: `gold.fact_cholera_deaths` (death metrics)

## Execution Time
~1-2 minutes

In [1]:
# ============================================
# ENVIRONMENT DETECTION & CONFIGURATION
# ============================================

import os
import sys
from pathlib import Path

# Auto-detect environment
IS_FABRIC = os.path.exists('/lakehouse/default')

if IS_FABRIC:
    print("🌐 Running in Microsoft Fabric")
    SILVER_TABLE_PATH = "/lakehouse/default/Tables/silver"
    GOLD_TABLE_PATH = "/lakehouse/default/Tables/gold"
    REFERENCE_PATH = "/lakehouse/default/Files/reference"
else:
    print("💻 Running locally")
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(project_root / 'src'))
    
    SILVER_TABLE_PATH = str(project_root / "data" / "silver_tables")
    GOLD_TABLE_PATH = str(project_root / "data" / "gold_tables")
    REFERENCE_PATH = str(project_root / "data" / "reference")
    
    # Create output directory
    Path(GOLD_TABLE_PATH).mkdir(parents=True, exist_ok=True)

print(f"Silver Path: {SILVER_TABLE_PATH}")
print(f"Gold Path: {GOLD_TABLE_PATH}")
print(f"Reference Path: {REFERENCE_PATH}")

💻 Running locally
Silver Path: D:\Projects\cholera-cdr-mvp\data\silver_tables
Gold Path: D:\Projects\cholera-cdr-mvp\data\gold_tables
Reference Path: D:\Projects\cholera-cdr-mvp\data\reference


In [2]:
# ============================================
# IMPORTS
# ============================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional
import logging

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Imports successful")

✅ Imports successful


In [3]:
# ============================================
# LOAD SILVER DATA
# ============================================

print("\n📂 Loading Silver layer data...\n")

try:
    if IS_FABRIC:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        
        df_silver_reports = spark.table("silver.report_summary").toPandas()
        df_silver_countries = spark.table("silver.country_weekly").toPandas()
    else:
        df_silver_reports = pd.read_parquet(Path(SILVER_TABLE_PATH) / "report_summary.parquet")
        df_silver_countries = pd.read_parquet(Path(SILVER_TABLE_PATH) / "country_weekly.parquet")
    
    print(f"✅ Loaded {len(df_silver_reports)} reports")
    print(f"✅ Loaded {len(df_silver_countries)} country records")
    
except Exception as e:
    logger.error(f"Error loading Silver data: {e}")
    raise


📂 Loading Silver layer data...

✅ Loaded 3 reports
✅ Loaded 6 country records


In [4]:
# ============================================
# LOAD REFERENCE DATA
# ============================================

print("\n📚 Loading reference data...\n")

try:
    # Load country codes
    df_country_ref = pd.read_csv(Path(REFERENCE_PATH) / "country_codes_iso3166.csv")
    print(f"✅ Loaded {len(df_country_ref)} country codes")
    
    # Load AU regions
    df_au_regions = pd.read_csv(Path(REFERENCE_PATH) / "au_regions.csv")
    print(f"✅ Loaded {len(df_au_regions)} AU region mappings")
    
    # Load epi week calendar
    df_epi_calendar = pd.read_csv(Path(REFERENCE_PATH) / "epi_week_calendar.csv")
    df_epi_calendar['start_date'] = pd.to_datetime(df_epi_calendar['start_date'])
    df_epi_calendar['end_date'] = pd.to_datetime(df_epi_calendar['end_date'])
    print(f"✅ Loaded {len(df_epi_calendar)} epi weeks")
    
except Exception as e:
    logger.error(f"Error loading reference data: {e}")
    raise


📚 Loading reference data...

✅ Loaded 53 country codes
✅ Loaded 5 AU region mappings
✅ Loaded 106 epi weeks


In [5]:
# ============================================
# CREATE DIMENSION: dim_country
# ============================================

print("\n🏗️ Creating dim_country...\n")

# Use country_ref directly (au_region already included in country_codes_iso3166.csv)
df_dim_country = df_country_ref.copy()

# Generate surrogate keys (1-based index)
df_dim_country['country_key'] = range(1, len(df_dim_country) + 1)

# Add SCD Type 1 columns
df_dim_country['is_current'] = True
df_dim_country['valid_from'] = datetime.now().date()
df_dim_country['valid_to'] = datetime(2099, 12, 31).date()  # Use 2099 instead of 9999
df_dim_country['created_at'] = datetime.now()
df_dim_country['updated_at'] = datetime.now()

# Select and order columns
dim_country_cols = [
    'country_key', 'country_code', 'country_name', 
    'who_region', 'au_region', 'population',
    'is_current', 'valid_from', 'valid_to',
    'created_at', 'updated_at'
]

df_dim_country = df_dim_country[dim_country_cols]

print(f"✅ Created dim_country with {len(df_dim_country)} countries")
print(f"   - Unique country codes: {df_dim_country['country_code'].nunique()}")
print(f"   - AU regions: {df_dim_country['au_region'].nunique()}")

# Display sample
print("\n📋 Sample dim_country:")
display(df_dim_country[['country_key', 'country_code', 'country_name', 'au_region', 'population']].head())


🏗️ Creating dim_country...

✅ Created dim_country with 53 countries
   - Unique country codes: 53
   - AU regions: 5

📋 Sample dim_country:


,country_key,country_code,country_name,au_region,population
0,1,DZA,Algeria,North,44400000
1,2,AGO,Angola,Central,31800000
2,3,BEN,Benin,West,12300000
3,4,BWA,Botswana,Southern,2630000
4,5,BFA,Burkina Faso,West,20900000


In [6]:
# ============================================
# CREATE DIMENSION: dim_date
# ============================================

print("\n🏗️ Creating dim_date...\n")

# Use epi week calendar as base
df_dim_date = df_epi_calendar.copy()

# Generate unique date_key using epi_year and epi_week (YYYYWW format)
df_dim_date['date_key'] = (df_dim_date['epi_year'] * 100 + df_dim_date['epi_week']).astype(int)
df_dim_date['date'] = df_dim_date['end_date']

# Add calendar attributes
df_dim_date['calendar_year'] = df_dim_date['date'].dt.year
df_dim_date['calendar_quarter'] = df_dim_date['date'].dt.quarter
df_dim_date['calendar_month'] = df_dim_date['date'].dt.month
df_dim_date['calendar_month_name'] = df_dim_date['date'].dt.strftime('%B')
df_dim_date['day_of_week'] = df_dim_date['date'].dt.dayofweek
df_dim_date['day_of_week_name'] = df_dim_date['date'].dt.strftime('%A')
df_dim_date['is_weekend'] = df_dim_date['day_of_week'].isin([5, 6])
df_dim_date['created_at'] = datetime.now()

# Select and order columns
dim_date_cols = [
    'date_key', 'date', 'epi_year', 'epi_week',
    'calendar_year', 'calendar_quarter', 'calendar_month', 'calendar_month_name',
    'day_of_week', 'day_of_week_name', 'is_weekend',
    'created_at'
]

df_dim_date = df_dim_date[dim_date_cols]

print(f"✅ Created dim_date with {len(df_dim_date)} dates")
print(f"   - Date range: {df_dim_date['date'].min()} to {df_dim_date['date'].max()}")
print(f"   - Epi years: {df_dim_date['epi_year'].min()} to {df_dim_date['epi_year'].max()}")
print(f"   - Unique date_keys: {df_dim_date['date_key'].nunique()}")

# Display sample
print("\n📋 Sample dim_date:")
display(df_dim_date[['date_key', 'date', 'epi_year', 'epi_week', 'calendar_month_name']].head())


🏗️ Creating dim_date...

✅ Created dim_date with 106 dates
   - Date range: 2024-01-07 00:00:00 to 2026-01-04 00:00:00
   - Epi years: 2024 to 2025
   - Unique date_keys: 106

📋 Sample dim_date:


,date_key,date,epi_year,epi_week,calendar_month_name
0,202401,2024-01-07,2024,1,January
1,202402,2024-01-14,2024,2,January
2,202403,2024-01-21,2024,3,January
3,202404,2024-01-28,2024,4,January
4,202405,2024-02-04,2024,5,February


In [7]:
# ============================================
# CREATE DIMENSION: dim_report
# ============================================

print("\n🏗️ Creating dim_report...\n")

df_dim_report = df_silver_reports[[
    'report_id', 'report_date', 'epi_year', 'epi_week',
    'data_quality_score', 'data_completeness_pct', 'source_file'
]].copy()

# Generate surrogate keys
df_dim_report['report_key'] = range(1, len(df_dim_report) + 1)

# Add report metadata
df_dim_report['report_type'] = 'Weekly'
df_dim_report['data_source'] = 'PDF'
df_dim_report['created_at'] = datetime.now()
df_dim_report['updated_at'] = datetime.now()

# Convert report_date to datetime if string
if df_dim_report['report_date'].dtype == 'object':
    df_dim_report['report_date'] = pd.to_datetime(df_dim_report['report_date'])

# Select and order columns
dim_report_cols = [
    'report_key', 'report_id', 'report_date', 'epi_year', 'epi_week',
    'report_type', 'data_source', 'quality_score', 'data_completeness_pct',
    'source_file', 'created_at', 'updated_at'
]

# Rename for consistency
df_dim_report = df_dim_report.rename(columns={'data_quality_score': 'quality_score'})

df_dim_report = df_dim_report[dim_report_cols]

print(f"✅ Created dim_report with {len(df_dim_report)} reports")
print(f"   - Average quality score: {df_dim_report['quality_score'].mean():.1f}")
print(f"   - Average completeness: {df_dim_report['data_completeness_pct'].mean():.1f}%")

# Display sample
print("\n📋 Sample dim_report:")
display(df_dim_report[['report_key', 'report_id', 'epi_year', 'epi_week', 'quality_score']].head())


🏗️ Creating dim_report...

✅ Created dim_report with 3 reports
   - Average quality score: 0.0
   - Average completeness: 100.0%

📋 Sample dim_report:


,report_key,report_id,epi_year,epi_week,quality_score
0,1,2025_wk06,2025,6,0.0
1,2,2025_wk07,2025,7,0.0
2,3,2025_wk08,2025,8,0.0


In [8]:
# ============================================
# CREATE FACT: fact_cholera_cases
# ============================================

print("\n🏗️ Creating fact_cholera_cases...\n")

# Start with country weekly data
df_fact_cases = df_silver_countries.copy()

# Join with dim_report to get report_key
df_fact_cases = df_fact_cases.merge(
    df_dim_report[['report_id', 'report_key']],
    on='report_id',
    how='left'
)

# Join with dim_country to get country_key
df_fact_cases = df_fact_cases.merge(
    df_dim_country[['country_code', 'country_key', 'population']],
    on='country_code',
    how='left'
)

# Convert report_date to datetime if string
if df_fact_cases['report_date'].dtype == 'object':
    df_fact_cases['report_date'] = pd.to_datetime(df_fact_cases['report_date'])

# Generate date_key from report_date
df_fact_cases['date_key'] = df_fact_cases['report_date'].dt.strftime('%Y%m%d').astype(int)

# Calculate metrics
df_fact_cases['new_cases'] = df_fact_cases['confirmed_cases']  # For MVP, new = confirmed
df_fact_cases['cumulative_cases'] = df_fact_cases.groupby('country_code')['confirmed_cases'].cumsum()

# Calculate incidence rate per 100,000 population
df_fact_cases['incidence_rate'] = (
    df_fact_cases['new_cases'] / df_fact_cases['population'] * 100000
).round(2)

# Attack rate (cumulative cases per 100,000)
df_fact_cases['attack_rate'] = (
    df_fact_cases['cumulative_cases'] / df_fact_cases['population'] * 100000
).round(2)

# Generate surrogate keys
df_fact_cases['case_key'] = range(1, len(df_fact_cases) + 1)

# Add audit column
df_fact_cases['created_at'] = datetime.now()

# Select and order columns
fact_cases_cols = [
    'case_key', 'report_key', 'country_key', 'date_key',
    'new_cases', 'cumulative_cases', 'confirmed_cases', 'suspected_cases',
    'attack_rate', 'incidence_rate',
    'created_at'
]

df_fact_cases = df_fact_cases[fact_cases_cols]

print(f"✅ Created fact_cholera_cases with {len(df_fact_cases)} records")
print(f"   - Total cases: {df_fact_cases['new_cases'].sum():,}")
print(f"   - Average incidence rate: {df_fact_cases['incidence_rate'].mean():.2f} per 100k")

# Display sample
print("\n📋 Sample fact_cholera_cases:")
display(df_fact_cases[['case_key', 'report_key', 'country_key', 'new_cases', 'incidence_rate']].head())


🏗️ Creating fact_cholera_cases...

✅ Created fact_cholera_cases with 6 records
   - Total cases: 2,053
   - Average incidence rate: 2.08 per 100k

📋 Sample fact_cholera_cases:


,case_key,report_key,country_key,new_cases,incidence_rate
0,1,1,53,355,2.40
1,2,1,52,245,1.26
2,3,2,53,409,2.76
3,4,2,52,372,1.92
4,5,3,53,429,2.90


In [9]:
# ============================================
# CREATE FACT: fact_cholera_deaths
# ============================================

print("\n🏗️ Creating fact_cholera_deaths...\n")

# Start with country weekly data
df_fact_deaths = df_silver_countries.copy()

# Join with dim_report to get report_key
df_fact_deaths = df_fact_deaths.merge(
    df_dim_report[['report_id', 'report_key']],
    on='report_id',
    how='left'
)

# Join with dim_country to get country_key
df_fact_deaths = df_fact_deaths.merge(
    df_dim_country[['country_code', 'country_key']],
    on='country_code',
    how='left'
)

# Convert report_date to datetime if string
if df_fact_deaths['report_date'].dtype == 'object':
    df_fact_deaths['report_date'] = pd.to_datetime(df_fact_deaths['report_date'])

# Generate date_key from report_date
df_fact_deaths['date_key'] = df_fact_deaths['report_date'].dt.strftime('%Y%m%d').astype(int)

# Calculate metrics
df_fact_deaths['new_deaths'] = df_fact_deaths['deaths']
df_fact_deaths['cumulative_deaths'] = df_fact_deaths.groupby('country_code')['deaths'].cumsum()

# Generate surrogate keys
df_fact_deaths['death_key'] = range(1, len(df_fact_deaths) + 1)

# Add audit column
df_fact_deaths['created_at'] = datetime.now()

# Select and order columns
fact_deaths_cols = [
    'death_key', 'report_key', 'country_key', 'date_key',
    'new_deaths', 'cumulative_deaths', 'cfr_percent',
    'created_at'
]

df_fact_deaths = df_fact_deaths[fact_deaths_cols]

print(f"✅ Created fact_cholera_deaths with {len(df_fact_deaths)} records")
print(f"   - Total deaths: {df_fact_deaths['new_deaths'].sum():,}")
print(f"   - Average CFR: {df_fact_deaths['cfr_percent'].mean():.2f}%")

# Display sample
print("\n📋 Sample fact_cholera_deaths:")
display(df_fact_deaths[['death_key', 'report_key', 'country_key', 'new_deaths', 'cfr_percent']].head())


🏗️ Creating fact_cholera_deaths...

✅ Created fact_cholera_deaths with 6 records
   - Total deaths: 46
   - Average CFR: 2.33%

📋 Sample fact_cholera_deaths:


,death_key,report_key,country_key,new_deaths,cfr_percent
0,1,1,53,7,1.97
1,2,1,52,9,3.67
2,3,2,53,8,1.96
3,4,2,52,5,1.34
4,5,3,53,11,2.56


In [10]:
# ============================================
# SAVE TO GOLD LAYER
# ============================================

print("\n💾 Saving to Gold layer...\n")

try:
    if IS_FABRIC:
        # Fabric: Use PySpark to write Delta tables
        spark_dim_country = spark.createDataFrame(df_dim_country)
        spark_dim_date = spark.createDataFrame(df_dim_date)
        spark_dim_report = spark.createDataFrame(df_dim_report)
        spark_fact_cases = spark.createDataFrame(df_fact_cases)
        spark_fact_deaths = spark.createDataFrame(df_fact_deaths)
        
        # Write to Delta tables (overwrite mode for MVP)
        spark_dim_country.write.format("delta").mode("overwrite").saveAsTable("gold.dim_country")
        spark_dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")
        spark_dim_report.write.format("delta").mode("overwrite").saveAsTable("gold.dim_report")
        spark_fact_cases.write.format("delta").mode("overwrite").saveAsTable("gold.fact_cholera_cases")
        spark_fact_deaths.write.format("delta").mode("overwrite").saveAsTable("gold.fact_cholera_deaths")
        
        print("✅ Delta tables created in Fabric Lakehouse")
        
    else:
        # Local: Save as Parquet
        # Convert datetime columns to strings
        for df, name in [
            (df_dim_country, 'dim_country'),
            (df_dim_date, 'dim_date'),
            (df_dim_report, 'dim_report'),
            (df_fact_cases, 'fact_cholera_cases'),
            (df_fact_deaths, 'fact_cholera_deaths')
        ]:
            df_save = df.copy()
            
            # Convert datetime columns to strings
            for col in df_save.columns:
                if df_save[col].dtype == 'datetime64[ns]' or 'datetime' in str(df_save[col].dtype):
                    df_save[col] = df_save[col].astype(str)
            
            # Save to parquet
            df_save.to_parquet(
                Path(GOLD_TABLE_PATH) / f"{name}.parquet",
                index=False,
                engine='pyarrow'
            )
            print(f"✅ Saved {name}.parquet ({len(df_save)} rows)")
        
        print(f"\n✅ All files saved to: {GOLD_TABLE_PATH}")
        
except Exception as e:
    logger.error(f"Error saving to Gold layer: {e}")
    raise

print("\n✅ Gold layer dimensional model complete!")


💾 Saving to Gold layer...

✅ Saved dim_country.parquet (53 rows)
✅ Saved dim_date.parquet (106 rows)
✅ Saved dim_report.parquet (3 rows)
✅ Saved fact_cholera_cases.parquet (6 rows)
✅ Saved fact_cholera_deaths.parquet (6 rows)

✅ All files saved to: D:\Projects\cholera-cdr-mvp\data\gold_tables

✅ Gold layer dimensional model complete!


## Validation & Testing

In [11]:
# ============================================
# VALIDATION & TESTING
# ============================================

print("\n🔍 Running validation checks...\n")

# Test 1: Dimension row counts
print("📊 Dimension Counts:")
print(f"  - dim_country: {len(df_dim_country)} countries")
print(f"  - dim_date: {len(df_dim_date)} dates")
print(f"  - dim_report: {len(df_dim_report)} reports")
assert len(df_dim_country) > 0, "dim_country is empty"
assert len(df_dim_date) > 0, "dim_date is empty"
assert len(df_dim_report) > 0, "dim_report is empty"
print("✅ All dimensions populated")

# Test 2: Fact row counts
print("\n📊 Fact Counts:")
print(f"  - fact_cholera_cases: {len(df_fact_cases)} records")
print(f"  - fact_cholera_deaths: {len(df_fact_deaths)} records")
assert len(df_fact_cases) > 0, "fact_cholera_cases is empty"
assert len(df_fact_deaths) > 0, "fact_cholera_deaths is empty"
print("✅ All facts populated")

# Test 3: Surrogate keys are unique
assert df_dim_country['country_key'].nunique() == len(df_dim_country), "Duplicate country_keys"
assert df_dim_date['date_key'].nunique() == len(df_dim_date), "Duplicate date_keys"
assert df_dim_report['report_key'].nunique() == len(df_dim_report), "Duplicate report_keys"
print("✅ All surrogate keys are unique")

# Test 4: Foreign key relationships
missing_report_keys = df_fact_cases[~df_fact_cases['report_key'].isin(df_dim_report['report_key'])]
assert len(missing_report_keys) == 0, f"Found {len(missing_report_keys)} orphaned report_keys in facts"
print("✅ All foreign keys valid")

# Test 5: Metrics calculated
assert df_fact_cases['incidence_rate'].notna().sum() > 0, "No incidence rates calculated"
assert df_fact_cases['attack_rate'].notna().sum() > 0, "No attack rates calculated"
print("✅ All metrics calculated")

# Test 6: Star schema integrity
print("\n⭐ Star Schema Summary:")
print(f"  - Dimensions: 3 (country, date, report)")
print(f"  - Facts: 2 (cases, deaths)")
print(f"  - Total fact records: {len(df_fact_cases) + len(df_fact_deaths)}")
print(f"  - Grain: One row per country per week")

print("\n✅ All validation checks passed!")


🔍 Running validation checks...

📊 Dimension Counts:
  - dim_country: 53 countries
  - dim_date: 106 dates
  - dim_report: 3 reports
✅ All dimensions populated

📊 Fact Counts:
  - fact_cholera_cases: 6 records
  - fact_cholera_deaths: 6 records
✅ All facts populated
✅ All surrogate keys are unique
✅ All foreign keys valid
✅ All metrics calculated

⭐ Star Schema Summary:
  - Dimensions: 3 (country, date, report)
  - Facts: 2 (cases, deaths)
  - Total fact records: 12
  - Grain: One row per country per week

✅ All validation checks passed!


## Next Steps

1. **Review dimensional model** above
2. **Verify star schema relationships** are correct
3. **Proceed to Notebook 04** for epidemiological analytics

## Outputs Created

**Dimensions:**
- `gold.dim_country` - Country master data with AU regions
- `gold.dim_date` - Date dimension with epi week calendar
- `gold.dim_report` - Report metadata with quality scores

**Facts:**
- `gold.fact_cholera_cases` - Case metrics with incidence/attack rates
- `gold.fact_cholera_deaths` - Death metrics with CFR

**Ready for Power BI!** ✅